# Tools

In simple terms, tools in an LLM are functions that an AI model can request to use when it needs to perform a task beyond simply generating text.

Think of an LLM as a smart person. The person can answer questions from what they know, but sometimes they need a calculator, a search engine, or a weather app. These are like tools.

## 1. What does “tools the model can request” mean?

Imagine you ask your AI assistant:

"What is the weather in Abuja?"

The AI may not know the current weather from its training data. It can request a weather tool to get current information.

You ask a question

"What is the weather in Abuja?"

LLM decides a tool is needed

Requests the weather function.

Tool runs

Fetches the current weather data.

LLM responds

Explains the weather using the returned data.

The important point is that the model usually requests the tool, but your application or framework executes it and sends the result back to the model.

## 2. What is a tool in an LLM?

A tool is usually a function with a name, a description, and defined inputs.

For example, a calculator tool could be:

```
def add_numbers(a: int, b: int) -> int:    """Add two numbers together."""    return a + b
```

Here:

- `add_numbers` is the tool's name.
- `a` and `b` are its inputs.
- The description tells the model what the tool does.
- `return a + b` performs the actual calculation.

If the model requests `add_numbers` with `a=5` and `b=3`, the function returns `8`.

## 3. How do you define a tool in LangChain?

Since you're learning LangChain, you can define a tool using the `@tool` decorator.

```
from langchain.tools import tool@tooldef add_numbers(a: int, b: int) -> int:    """Add two numbers together."""    return a + bprint(add_numbers.invoke({"a": 5, "b": 3}))
```

Output:

```
8
```

Notice that this example executes the tool directly. It doesn't yet involve an LLM deciding when to use it.

## 4. How does an LLM request the tool?

You can provide a tool to a model using LangChain:

```
from langchain.chat_models import init_chat_modelfrom langchain.tools import toolimport osfrom dotenv import load_dotenvload_dotenv()model = init_chat_model(    "groq:openai/gpt-oss-120b",    api_key=os.getenv("GROQ_API_KEY"))@tooldef add_numbers(a: int, b: int) -> int:    """Add two numbers together."""    return a + bmodel_with_tools = model.bind_tools([add_numbers])response = model_with_tools.invoke(    "What is 25 + 17? Use the calculator tool.")print(response.tool_calls)
```

The model can return a tool call request containing the tool name and arguments. In this example, it might request `add_numbers` with `a=25` and `b=17`.

Binding a tool makes it available to the model, but it doesn't necessarily execute it automatically. Your code or a LangChain agent must execute the request and return the result.

## 5. Common tools used with LLMs

| Tool             | What it does                        |
| ---------------- | ----------------------------------- |
| Calculator       | Performs calculations               |
| Web search       | Retrieves current information       |
| Database query   | Retrieves records from a database   |
| Weather function | Gets weather information            |
| Python function  | Performs a specific programmed task |
| File search      | Finds information in documents      |

Remember: The LLM is the decision-maker, the tool is the capability, and your Python function is what performs the task. A model can also answer directly without requesting any tool when one isn't needed.

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON Schema)
2. A function or continue to exceute

In [3]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY")
)

response = model.invoke("Why do parrots talk?")

print(response.content)


Parrots don’t **talk** in the same way humans do— they don’t understand language or generate original sentences. What they do is **vocal mimicry**, and several biological and social factors make them especially good at it.

---

## 1. Anatomy that enables mimicry  

| Feature | How it helps |
|---------|--------------|
| **Syrinx** (the bird’s voice box) | Located at the base of the trachea, the syrinx has highly flexible membranes and muscles. In parrots these structures are especially developed, allowing precise control over pitch, tone, and rhythm. |
| **Tongue & beak** | A muscular, mobile tongue and a strong, articulated beak let them shape sounds in ways most birds cannot. |
| **Auditory system** | Parrots have excellent hearing, especially for the frequencies used in human speech, so they can pick up subtle nuances and reproduce them. |

---

## 2. Evolutionary and social reasons  

| Reason | Explanation |
|--------|-------------|
| **Flock cohesion** | In the wild, parrots liv

In [4]:
# creating a tool
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather]) 

In [5]:
# to call the model
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks weather in Boston. Need to call function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_e0511636-bdb3-4dfe-8cc0-84658cbf2e1e', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 127, 'total_tokens': 172, 'completion_time': 0.094311154, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.03053952, 'prompt_tokens_details': None, 'queue_time': 0.310623755, 'total_time': 0.124850674}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a159c1-ff69-7f52-9440-6a7590f49f1d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_e0511636-bdb3-4dfe-8cc0-84658cbf2e1e', 'type': 'tool_call'}] invalid_tool_calls=[] u

### Tool Execution Loops

In [11]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)



It's sunny in Boston right now.


In [12]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_de413247-b2e2-46d0-b40f-80a905c68029', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 126, 'total_tokens': 169, 'completion_time': 0.091189901, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.009029679, 'prompt_tokens_details': None, 'queue_time': 0.298118297, 'total_time': 0.10021958}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a7bb72c24', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a159e6-8f19-7590-b0ba-09c95f0e8599-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_de413247-b2e2-46d0-b

In [13]:
# Call the regular API
import requests

def get_weather(city: str) -> dict:
    """Get current weather information for a city."""

    url = "https://wttr.in/{0}?format=j1".format(city)

    response = requests.get(url, timeout=15)
    response.raise_for_status()

    data = response.json()
    current = data["current_condition"][0]

    return {
        "city": city,
        "temperature_c": current["temp_C"],
        "description": current["weatherDesc"][0]["value"]
    }

print(get_weather("Abuja"))

{'city': 'Abuja', 'temperature_c': '34', 'description': 'Sunny'}


In [14]:
# Turn the API function into a LangChain tool
from langchain.tools import tool

@tool
def weather_tool(city: str) -> dict:
    """Get the current weather for a city."""
    return get_weather(city)

In [15]:
# Let your AI model use the API

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv()

model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY")
)

model_with_tools = model.bind_tools([weather_tool])

response = model_with_tools.invoke(
    "What is the current weather in Abuja?"
)

print(response.tool_calls)

[{'name': 'weather_tool', 'args': {'city': 'Abuja'}, 'id': 'fc_bffc51d0-8050-4c0f-becf-45591bd6d88a', 'type': 'tool_call'}]
